# Initial Setup

In [41]:
import torch

if not torch.cuda.is_available():
  print("Warning: No GPU found. Please add GPU to your notebook")

 # use only gpu since llama vision requires 12gb of ram


In [42]:
%%capture
!sudo apt-get install poppler-utils

In [43]:
!curl https://ollama.ai/install.sh | sh

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 13269    0 13269    0     0  61706      0 --:--:-- --:--:-- --:--:-- 62004
>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading Linux amd64 bundle
############################################################################################# 100.0%
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [67]:
!ollama serve > server.log 2>&1 &

In [45]:
%%capture
llava_model = "llava:7b"
llama_vision_model = "llama3.2-vision"


!ollama pull {llava_model}
!ollama pull {llama_vision_model}



In [71]:
!ollama ls

NAME                                   ID              SIZE      MODIFIED       
erwan2/DeepSeek-Janus-Pro-7B:latest    e877a212a6a7    4.2 GB    39 seconds ago    
llama3.2-vision:latest                 085a1fdae525    7.9 GB    37 minutes ago    
llava:7b                               8dd30f6b0cb1    4.7 GB    37 minutes ago    


In [47]:
%%capture
!pip install py-zerox litellm==1.57.4 nest_asyncio camelot-py

In [48]:
# download the pdf
!wget -O table.pdf https://ada.nv.gov/uploadedFiles/adanewnvgov/content/home/features/ADAClasses/Merged%20Cell%20Table.pdf


--2025-02-03 20:54:03--  https://ada.nv.gov/uploadedFiles/adanewnvgov/content/home/features/ADAClasses/Merged%20Cell%20Table.pdf
Resolving ada.nv.gov (ada.nv.gov)... 167.154.11.65
Connecting to ada.nv.gov (ada.nv.gov)|167.154.11.65|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 136964 (134K) [application/pdf]
Saving to: ‘table.pdf’

table.pdf           100%[===================>] 133.75K  --.-KB/s    in 0.1s    

2025-02-03 20:54:04 (1.22 MB/s) - ‘table.pdf’ saved [136964/136964]



# Standard python library camelot-py

In [49]:
import camelot
tables = camelot.read_pdf('/content/table.pdf')

for table in tables:
  print(table.df.to_markdown())

|    | 0                | 1                | 2             |
|---:|:-----------------|:-----------------|:--------------|
|  0 | Health Insurance |                  |               |
|  1 | PPO              | Office Copay     | $10.00        |
|  2 |                  | O                | $1500 Single  |
|    |                  | ut- of -pocket   | $2750 Family  |
|    |                  | Maximum          |               |
|  3 |                  | F                | $             |
|    |                  | amily            | 350 monthly   |
|  4 | HMO              | O                | $             |
|    |                  | ffice Copay      | 5.00 Single   |
|  5 |                  | O                | $1000 Single  |
|    |                  | ut - of - pocket | $2250 Family  |
|  6 |                  | F                | $             |
|    |                  | amily            | 250 Monthly   |


# Bug Fix as reported [here](https://github.com/getomni-ai/zerox/issues/106)

In [50]:
import litellm.litellm_core_utils.prompt_templates.factory as factory
import litellm.llms.ollama.completion.transformation as transformation
import litellm

import requests

def ollama_query_online_is_vision_model(model:str):
    if "/" not in model:
        model_fullname = f"library/{model}"
    else:
        model_fullname = model
    url = f"https://ollama.com/{model_fullname}"
    response = requests.get(url).text
    ret = ">vision</span>" in response
    return ret

def ollama_query_offline_is_vision_model(model:str):
    known_vision_models = ['moondream', 'jyan1/paligemma-mix-224','benzie/llava-phi-3' , 'mskimomadto/chat-gph-vision',  'knoopx/llava-phi-2', 'llava']
    for it in known_vision_models:
        if it in model:
            return True
    if "vision" in model:
        return True
    if model.lower().endswith("vlm"):
        return True
    if model.lower().endswith('v'):
        return True

def ollama_is_vision_model(model:str, online:bool = False):
    factory.verbose_logger.info("Checking if model is vision model: {}".format(model))
    factory.verbose_logger.info("Checking method online: {}".format(str(online)))
    if online:
        ret = ollama_query_online_is_vision_model(model)
    else:
        ret = ollama_query_offline_is_vision_model(model)
    factory.verbose_logger.info("Checking vision support result: {}".format(str(ret)))
    return ret

def ollama_pt(
    model, messages
) -> factory.Union[
    str, factory.OllamaVisionModelObject
]:  # https://github.com/ollama/ollama/blob/af4cf55884ac54b9e637cd71dadfe9b7a5685877/docs/modelfile.md#template
    if "instruct" in model:
        prompt = factory.custom_prompt(
            role_dict={
                "system": {"pre_message": "### System:\n", "post_message": "\n"},
                "user": {
                    "pre_message": "### User:\n",
                    "post_message": "\n",
                },
                "assistant": {
                    "pre_message": "### Response:\n",
                    "post_message": "\n",
                },
            },
            final_prompt_value="### Response:",
            messages=messages,
        )
    elif ollama_is_vision_model(model):
        prompt = ""
        images = []
        for message in messages:
            if isinstance(message["content"], str):
                prompt += message["content"]
            elif isinstance(message["content"], list):
                # see https://docs.litellm.ai/docs/providers/openai#openai-vision-models
                for element in message["content"]:
                    if isinstance(element, dict):
                        if element["type"] == "text":
                            prompt += element["text"]
                        elif element["type"] == "image_url":
                            base64_image = factory.convert_to_ollama_image(
                                element["image_url"]["url"]
                            )
                            images.append(base64_image)
        return {"prompt": prompt, "images": images}
    else:
        prompt = ""
        for message in messages:
            role = message["role"]
            content = message.get("content", "")

            if "tool_calls" in message:
                tool_calls = []

                for call in message["tool_calls"]:
                    call_id: str = call["id"]
                    function_name: str = call["function"]["name"]
                    arguments = factory.json.loads(call["function"]["arguments"])

                    tool_calls.append(
                        {
                            "id": call_id,
                            "type": "function",
                            "function": {"name": function_name, "arguments": arguments},
                        }
                    )

                prompt += f"### Assistant:\nTool Calls: {factory.json.dumps(tool_calls, indent=2)}\n\n"

            elif "tool_call_id" in message:
                prompt += f"### User:\n{message['content']}\n\n"

            elif content:
                prompt += f"### {role.capitalize()}:\n{content}\n\n"

    return prompt

del factory.ollama_pt
del transformation.ollama_pt
del litellm.main.ollama_pt

setattr(factory, "ollama_pt", ollama_pt)
setattr(transformation, "ollama_pt", ollama_pt)
setattr(litellm.main, "ollama_pt", ollama_pt)

In [51]:
import litellm

litellm.supports_vision = lambda *args, **kwargs: True
litellm.check_valid_key = lambda *args, **kwargs: True

# Implementation

In [52]:
from pyzerox import zerox
import os
import json
import asyncio
import os
import asyncio
import nest_asyncio

nest_asyncio.apply() # we beed nested asyncio in google colab since ... explain

os.environ['OLLAMA_API_BASE'] = "http://localhost:11434"

pdf_path = "/content/table.pdf"
output_path = "./"
kwargs = {}



In [53]:
async def call_model(model, file, custom_system_prompt, kwargs):
  select_pages = None ## None for all, but could be int or list(int) page numbers (1 indexed)
  model = f"ollama/{model}"

  result = await zerox(file_path=pdf_path, model=model, output_dir=output_path,
                        custom_system_prompt=custom_system_prompt,select_pages=select_pages,validate_vision_capability=False, **kwargs)

  return result.pages[0].content


# First try, no custom prompt

In [36]:
result = asyncio.run(call_model(model=llama_vision_model, file=pdf_path, custom_system_prompt=None, kwargs=kwargs))
print(result)

**Chapter 7: Insurance Plans**

In this chapter, we will explore how to handle table structure with complex tables. We will also discuss best practices for table structure and learn how to set headers with different heading levels.

**Table Structure**

The following is an example of a complex table with multiple rows and columns:

| **Health Insurance** | **PPO** |
| --- | --- |
| **Office Copay** | $10.00 |
| **Out-of-pocket Maximum** | Single: $1500, Family: $2750 |
| **Family** | Monthly: $350 |

| **HMO** |
| --- |
| **Office Copay** | $5.00 |
| **Out-of-pocket Maximum** | Single: $1000, Family: $2250 |
| **Family** | Monthly: $250 |

**Best Practices for Table Structure**

* Use clear and concise headings
* Use tables to organize data in a logical and easy-to-read format
* Use different heading levels (e.g. H1, H2, H3) to create a hierarchy of information

**Setting Headers with Different Heading Levels**

To set headers with different heading levels, use the following syntax:

*

In [58]:
result = asyncio.run(call_model(model=llava_model, file=pdf_path, custom_system_prompt=None, kwargs=kwargs))
print(result)

 ```markdown
    # Chapter 7: Business Intelligence Dashboards - Power BI
    The purpose of this chapter is to provide an introduction to using Microsoft Power BI for creating interactive business intelligence dashboards.
    This is a **Chapter Title**.

    ## Table of Contents
    1. Overview
    2. Installation Process
    3. Data Source Examples
    4. Dashboard Design Guidelines
    5. Creating Interactive Elements
    6. Publishing and Sharing Dashboards

    ```markdown
    This chapter provides an overview of creating interactive business intelligence dashboards using Microsoft Power BI.
    ```

    It covers the following topics:

    - Installation Process
    - Data Source Examples
    - Dashboard Design Guidelines
    - Creating Interactive Elements
    - Publishing and Sharing Dashboards

    This chapter focuses on:

    - Installing Power BI Desktop
    - Setting up data sources
    - Designing interactive dashboards
    - Publishing and sharing dashboards
    
    To

# Second try, customize system prompt

In [73]:
custom_system_prompt ="For the below PDF page, extract tables and text in markdown format. You will extract multiple tables. If the tables have an unusual or complex layout, you are allowed to reorganize them, but you must ensure that all information from the original tables is preserved. Return only the markdown with no explanation text. Tables may be empty, do not exclude them. Do not exclude any content from the page.Do not write anything else. \n\n"
#custom_system_prompt = "You are a model tasked with converting PDFs into Markdown format. Your main goal is to extract and convert all the tables present in the PDF into Markdown tables. If the tables have an unusual or complex layout, you are allowed to reorganize them, but you must ensure that all information from the original tables is preserved. Your output should be the complete Markdown representation of the PDF, with a particular emphasis on accurately converting the tables. Do not include any comments, explanations, or additional text—only the Markdown content of the PDF."


In [56]:
result = asyncio.run(call_model(model=llama_vision_model, file=pdf_path, custom_system_prompt=custom_system_prompt, kwargs=kwargs))
print(result)

/usr/local/lib/python3.11/dist-packages/pyzerox/models/modellitellm.py:52: UserWarning: 
    Custom system prompt was provided which overrides the default system prompt. We assume that you know what you are doing.  
    . Default prompt for zerox is:
 
    Convert the following PDF page to markdown.
    Return only the markdown with no explanation text.
    Do not exclude any content from the page.
    
  warnings.warn(f"{Messages.CUSTOM_SYSTEM_PROMPT_WARNING}. Default prompt for zerox is:\n {DEFAULT_SYSTEM_PROMPT}")


# **Chapter 7: Insurance Plans**

## Table 1: Health Insurance PPO and HMO Plans

| **Health Insurance** | **PPO** | **HMO** |
| --- | --- | --- |
| **Office Copay** | $10.00 | $5.00 |
| **Out-of-Pocket Maximum** | $1500 | $1000 |
| **Family Plan** | $2750 | $2250 |

## Table 2: Insurance Plans

| **Insurance Plan** | **Description** |
| --- | --- |
| **PPO** | Preferred Provider Organization |
| **HMO** | Health Maintenance Organization |

# **7.1 Insurance Plans**

This is an example of a complex table with different heading styles covering multiple rows and columns in a single table.

## Table 3: Insurance Plan Details

| **Insurance Plan** | **Description** | **Cost** |
| --- | --- | --- |
| **PPO** | Preferred Provider Organization | $10.00 |
| **HMO** | Health Maintenance Organization | $5.00 |

# **Chapter 7**

In this section, we will cover how to handle table structure with complex tables. This includes an overview of the best practices for table structure and how to set heade

In [63]:
custom_system_prompt ="You are a model tasked with converting PDFs into Markdown format. Your main goal is to extract and convert all the tables present in the PDF into Markdown tables. If the tables have an unusual or complex layout, you are allowed to reorganize them, but you must ensure that all information from the original tables is preserved. Your output should be the complete Markdown representation of the PDF, with a particular emphasis on accurately converting the tables. Do not include any comments, explanations, or additional text—only the Markdown content of the PDF."

result = asyncio.run(call_model(model=llava_model, file=pdf_path, custom_system_prompt=custom_system_prompt, kwargs=kwargs))
print(result)

/usr/local/lib/python3.11/dist-packages/pyzerox/models/modellitellm.py:52: UserWarning: 
    Custom system prompt was provided which overrides the default system prompt. We assume that you know what you are doing.  
    . Default prompt for zerox is:
 
    Convert the following PDF page to markdown.
    Return only the markdown with no explanation text.
    Do not exclude any content from the page.
    
  warnings.warn(f"{Messages.CUSTOM_SYSTEM_PROMPT_WARNING}. Default prompt for zerox is:\n {DEFAULT_SYSTEM_PROMPT}")


 Certainly! Here's the Markdown representation of the table in the image:

```
| Item                         | Quantity   |
|--------------------------------|--------------|
| Office Supplies                    | 1           |
| File Cabinet                     | 1           |
| Worktable                          | 2           |
| Chair                               | 1           |
| Coffee Maker                      | 1           |
| Printer/Scanner                       | 1           |
| Desk Lamp                           | 1           |
| Computer Monitor                   | 1           |
| Keyboard                              | 1           |
| Mouse                                 | 1           |
```

And here's the Markdown representation of the second table in the image:

```
| Item                               | Quantity   |
|----------------------------------|--------------|
| Office Supplies                            | $50/month     |
| Printer/Scanner                    